# Chapter 4 : Parallelization
# What is Parallelization ?

# Parallelization means doing multiple tasks at the same time instead of waiting for one to finish before starting the next.

# Doing API Authentication

In [206]:
from google import genai
from google.genai import types

# Developer TODO: Replace YOUR_API_KEY with your API key.
API_KEY = "YOUR API KEY"

client = genai.Client(
    vertexai=False, api_key=API_KEY
)

In [207]:
import os

os.environ["GEMINI_API_KEY"] = "YOUR API KEY"

In [228]:
chat = client.chats.create(model="gemini-3.6-flash")

In [211]:
# Step 1 — Build a simple router

In [230]:
from google.adk.agents import LlmAgent, ParallelAgent, SequentialAgent
from google.adk.tools import google_search
GEMINI_MODEL="gemini-3.6-flash"
import os
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
import os

# # Ensure your working Gemini API key is configured
# os.environ["GEMINI_API_KEY"] = API_KEY
# if "GOOGLE_GENAI_USE_VERTEXAI" in os.environ:
#     del os.environ["GOOGLE_GENAI_USE_VERTEXAI"]

# # Clean up GOOGLE_API_KEY to prevent it from hijacking the Gemini client authentication
# if "GOOGLE_API_KEY" in os.environ:
#     del os.environ["GOOGLE_API_KEY"]
# if "GOOGLE_CSE_ID" in os.environ:
#     del os.environ["GOOGLE_CSE_ID"]

In [231]:
# Researcher 1: Renewable Energy

researcher_agent_1 = LlmAgent(
    name = "RenewableEnergyResearcher",
    model = GEMINI_MODEL,
    instruction= """ You are an AI Research Assistant specializing in energy. Research the latest advancement in 'renewable energy sources
    Summarize your key findings concisely""",
    description = "Research renewable energy sources",
    tools = [], # Removed google_search tool to avoid credentials error
    # store result in state for the merger agent
    output_key = "renewable_energy_agent"
)

In [232]:
# Reseacher 2: Electric Vehicles

reseacher_agent_2 = LlmAgent(
    name = "EV_Researcher",
    model = GEMINI_MODEL,
    instruction= """ You are an AI Research Assistant specializing in energy. Research the latest advancement in 'Electric Vehicles'
    Summarize your key findings concisely""",
    description = "Research Electric Vehicles",
    tools = [], # Removed google_search tool to avoid credentials error
    # store result in state for the merger agent
    output_key = "ev_technology_result"
)

In [233]:
# Reseacher 3: Carbon Capture

researcher_agent_3 = LlmAgent(
    name = "Carbon_Capture_Researcher",
    model = GEMINI_MODEL,
    instruction= """ You are an AI Research Assistant specializing in energy. Research the latest advancement in 'Carbon Capture'
    Summarize your key findings concisely""",
    description = "Research Carbon Capture",
    tools = [], # Removed google_search tool to avoid credentials error
    # store result in state for the merger agent
    output_key = "carbon_capture_result"
)

In [234]:
## Create the parallel agent ( Runs all researcher concurrently)
# it finishes once all researchers have completed and stored their results in state

In [235]:
parallel_reseach_agent = ParallelAgent(
    name = "Parallel_Researcher",
    sub_agents = [researcher_agent_1, reseacher_agent_2, researcher_agent_3],
    description = "Runs Multiple Research agents in parallel together information ")

/tmp/ipykernel_4427/3957745359.py:1: DeprecationWarning: ParallelAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  parallel_reseach_agent = ParallelAgent(


In [236]:
# Define the merger agent ( Runs after the parallel agent )

# This agent takes the results stored in the session state by the parallel agents and sythesizes them into a single structured response with attributions.

merger_agent = LlmAgent(
    name = "SynthesisAgent",
    model = GEMINI_MODEL,
    instruction = """ You are an AI assistant responsible for combining research findings into a structured report. YOur primary task is to syntehsie the following reseach summaries . Structure your response using headings for each topic.

    ** Input Summaries : **
    ** Rebewable Energy :**
    {renewable_energy_agent}
    ** Electric Vehicles :**
    {ev_technology_result}
    ** Carbon Capture :**
    {carbon_capture_result}


** Output Format : ** """,

    description = "Combine reseach findings from parallel agents into a structured, cited report, strictly grounded on provided inputs "




)

In [237]:
# Create the Sequential Agent (Orchestrates the overall flow)
# This is the main agent theat will be run. It first executes the Paralel Agent

sequential_pipeline_agent = SequentialAgent(
    name = "ReseachAndSysnthesisPipeline",
    sub_agents = [parallel_reseach_agent, merger_agent],
    description = "Coordinates parallel research and synthesizes the results")

root_agent = sequential_pipeline_agent

/tmp/ipykernel_4427/4127827195.py:4: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  sequential_pipeline_agent = SequentialAgent(


In [238]:
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(root_agent)

await runner.session_service.create_session(
    app_name=runner.app_name,
    user_id="user",
    session_id="session"
)


Session(id='session', app_name='InMemoryRunner', user_id='user', state={}, events=[], last_update_time=1790115157.0500288)

In [241]:
async for event in runner.run_async(
    user_id="user",
    session_id="session",
    new_message=types.Content(
        role="user",
        parts=[
            types.Part(
                text="Research the latest advancements in renewable energy, electric vehicles, and carbon capture."
            )
        ]
    )
):
    if event.is_final_response():
        print(event.content.parts[0].text)

Here is a concise summary of the latest advancements in **Electric Vehicles (EVs)**, along with key updates in **Renewable Energy** and **Carbon Capture**.

---

### 1. Electric Vehicles (EVs) *(Primary Focus)*

* **Solid-State & Sodium-Ion Battery Breakthroughs:** 
  * **Solid-State:** Major automakers and battery manufacturers are piloting semi-solid and solid-state batteries, offering double the energy density of traditional Li-ion, 10–15 minute fast charging, and significantly reduced fire risks.
  * **Sodium-Ion (Na-ion):** Mass production of sodium-ion batteries has begun for entry-level EVs, completely removing reliance on expensive lithium, cobalt, and nickel.
* **Next-Gen Charging Architectures (800V/1000V & V2G):**
  * Widespread adoption of **800V ultra-fast charging platforms** enables 10–80% charge times in under 15 minutes.
  * **Vehicle-to-Grid (V2G)** and bidirectional charging are being built natively into new models, turning EVs into distributed energy storage units f

_ResourceExhaustedError: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/google-gemini/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.6-flash\nPlease retry in 54.268612889s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_cont


# --- THE END ---

